[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elyssephillips/embryo_image_analysis/blob/main/pipelines/xenium/oocyte_transcript_counts_colab.ipynb)

# Oocyte transcript counting (Colab version)

Counts how many transcript dots (of one gene) fall inside the oocyte versus in the
surrounding granulosa cells for one ROI of a follicle exported as a PNG.

When you run the Parameters cell below, you'll be prompted to upload 3 PNGs exported for the same view: an annotations file (granulosa cell outlines, transparent background), a transcripts file (transcript dots, transparent background), and a cell layer file (the fluorescent image, for overlays). All three must be the same pixel size.

1. Load both PNGs and confirm they're aligned
2. Measure the exported scale bar to convert pixels to microns
3. Find granulosa layer 1 (the ring bordering the oocyte) by color, then close its small
   gaps and fill it in to get the oocyte's boundary
4. Find every transcript dot by color, and check which ones fall inside that boundary

This is the Colab-only version -- no local file path, no environment branching. For a version that reads PNGs from a local folder instead, see `oocyte_transcript_counts.ipynb` in this same folder.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from scipy.ndimage import binary_closing, binary_fill_holes, distance_transform_edt
from skimage.morphology import disk
from skimage.measure import label, regionprops, find_contours
import matplotlib.pyplot as plt

%matplotlib inline

## Parameters

Per sample

In [ ]:
# -- where the exported PNGs are --
from google.colab import files
print("Select the 3 exported PNGs to upload: annotations_layer.png, "
      "transcripts_layer.png, Cell_layer.png")
files.upload()  # uploads land in /content, keeping their original filenames
INPUT_DIR = Path("/content")
ANNOTATIONS_FILE = INPUT_DIR / "annotations_layer.png"
TRANSCRIPTS_FILE = INPUT_DIR / "transcripts_layer.png"
CELL_LAYER_FILE = INPUT_DIR / "Cell_layer.png"  # morphology image (DAPI etc.) with cell outlines
OUTPUT_DIR = INPUT_DIR / "analysis"

# -- granulosa layer outline colors --
LAYER1_RGB = (255, 100, 100)  # inner ring, borders the oocyte
LAYER1_COLOR_TOLERANCE = 20   # how far off the exact RGB a pixel can be and still count
LAYER2_RGB = (69, 139, 255)   # outer ring
LAYER2_COLOR_TOLERANCE = 20
ALPHA_MIN = 51                # out of 255; drops faint edge pixels

# -- transcript dot color --
GREEN_DOMINANCE_MIN = 20      # green channel must exceed red and blue by at least this much

# -- geometry --
CLOSING_RADIUS_UM = 3.0       # bridges small gaps in the granulosa layer 1 outline
SCALE_BAR_UM = 50             
PX_PER_UM_OVERRIDE = None     # set this manually if scale-bar auto-detection ever fails

# -- plot colors: one consistent, colorblind-safe palette used in every figure below.
# Same three hues always mean the same three regions, everywhere in this notebook.
SURFACE = "#fcfcfb"       # chart background (near-white)
INK = "#0b0b0b"           # primary text/labels
INK_MUTED = "#898781"     # axis ticks, and the "background" (uncategorized) region
GRID = "#e1e0d9"          # gridlines / spines
OOCYTE_COLOR = "#2a78d6"        # blue
LAYER1_COLOR = "#eb6834"        # orange
LAYER2_COLOR = "#1baf7a"        # aqua

# These two PNGs (annotations_layer.png, transcripts_layer.png) have a transparent
# background -- shown on a light surface, faint content (like the green transcript
# dots) disappears. Panels that display those raw layers directly use this dark
# background instead; panels showing an actual chart (bars, lines) or an opaque image
# (Cell_layer.png, binary masks) keep using SURFACE/INK above.
PNG_BG = "black"
PNG_INK = "white"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sample_id = INPUT_DIR.name
sample_id

## Load images

Both PNGs load as RGBA arrays: red, green, blue, and alpha (transparency) per pixel.
The background is fully transparent (alpha=0); drawn content (outlines, dots) has
nonzero alpha.


In [ ]:
annotations_rgba = np.array(Image.open(ANNOTATIONS_FILE).convert("RGBA"))
transcripts_rgba = np.array(Image.open(TRANSCRIPTS_FILE).convert("RGBA"))

print(f"annotations layer: {annotations_rgba.shape[1]} x {annotations_rgba.shape[0]} px")
print(f"transcripts layer: {transcripts_rgba.shape[1]} x {transcripts_rgba.shape[0]} px")

assert annotations_rgba.shape[:2] == transcripts_rgba.shape[:2], \
    "annotations and transcripts layers have different dimensions -- exports must be aligned."
print("dimensions match, layers are aligned")

# Visual check: does this look like what you expect?
# Both PNGs have a transparent background -- show them on the chart surface color so
# nothing (like the faint transcript dots) disappears against a mismatched background.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(PNG_BG)
for ax in axes:
    ax.set_facecolor(PNG_BG)
axes[0].imshow(annotations_rgba)
axes[0].set_title("annotations layer", color=PNG_INK)
axes[0].axis("off")
axes[1].imshow(transcripts_rgba)
axes[1].set_title("transcripts layer", color=PNG_INK)
axes[1].axis("off")
plt.show()

## Find the scale bar

Xenium Explorer includes a scale bar, which we can use so every later measurement (closing radius, area) can be
reported in real units instead of pixels that would change with export zoom.
.


In [ ]:
def find_px_per_um(rgba, scale_bar_um, search_right_fraction=0.5):
    red = rgba[..., 0].astype(int)
    green = rgba[..., 1].astype(int)
    blue = rgba[..., 2].astype(int)
    alpha = rgba[..., 3]

    width = rgba.shape[1]
    is_scale_bar_pixel = (
        (alpha == 255)
        & (np.abs(red - 240) < 10) & (np.abs(green - 240) < 10) & (np.abs(blue - 240) < 10)
    )
    is_scale_bar_pixel[:, : int(width * search_right_fraction)] = False

    _, xs = np.where(is_scale_bar_pixel)
    if xs.size == 0:
        raise RuntimeError("Could not auto-detect scale bar -- set PX_PER_UM_OVERRIDE above.")
    scale_bar_length_px = xs.max() - xs.min()
    return scale_bar_length_px / scale_bar_um


px_per_um = PX_PER_UM_OVERRIDE or find_px_per_um(transcripts_rgba, SCALE_BAR_UM)
print(f"scale: {px_per_um:.3f} px/um")


Build the oocyte mask

`granulosa_layer1`is the outline of 
granulosa cells in that layer, which can have gaps between cells. Close these gaps so we can create a continous granulosa border. 

1. **Color-match** every pixel close enough to the layer-1 color to call it "outline"
2. **Close** the small gaps (morphological closing = briefly fatten every line, then
   shrink back down -- enough to bridge the small spaces between cells
3. **Fill** everything now enclosed by the granulosa ring, then subtract
   it back out, leaving just the oocyte interior


In [ ]:
red = annotations_rgba[..., 0].astype(int)
green = annotations_rgba[..., 1].astype(int)
blue = annotations_rgba[..., 2].astype(int)
alpha = annotations_rgba[..., 3]

granulosa_layer1_mask = (
    (alpha >= ALPHA_MIN)
    & (np.abs(red - LAYER1_RGB[0]) <= LAYER1_COLOR_TOLERANCE)
    & (np.abs(green - LAYER1_RGB[1]) <= LAYER1_COLOR_TOLERANCE)
    & (np.abs(blue - LAYER1_RGB[2]) <= LAYER1_COLOR_TOLERANCE)
)
print(f"granulosa layer 1 outline: {granulosa_layer1_mask.sum():,} px matched color {LAYER1_RGB}")

fig = plt.figure(figsize=(7, 5))
fig.patch.set_facecolor(SURFACE)
plt.imshow(granulosa_layer1_mask, cmap="gray")
plt.title("granulosa layer 1 -- color-matched pixels", color=INK)
plt.axis("off")
plt.show()

###Fill gaps between cells


In [ ]:
closing_radius_px = CLOSING_RADIUS_UM * px_per_um
closed_ring = binary_closing(granulosa_layer1_mask, structure=disk(max(1, round(closing_radius_px))))

n_pixels_added = (closed_ring & ~granulosa_layer1_mask).sum()
print(f"closing added {n_pixels_added:,} px to bridge gaps in the outline")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(SURFACE)
axes[0].imshow(granulosa_layer1_mask, cmap="gray")
axes[0].set_title("before closing", color=INK)
axes[0].axis("off")
axes[1].imshow(closed_ring, cmap="gray")
axes[1].set_title("after closing", color=INK)
axes[1].axis("off")
plt.show()

In [ ]:
filled_disk = binary_fill_holes(closed_ring)
oocyte_mask = filled_disk & ~closed_ring

# Safety check: keep only the largest enclosed region, there are sometimes other gaps still present.
labeled_regions = label(oocyte_mask)
if labeled_regions.max() == 0:
    raise RuntimeError("No enclosed region found -- granulosa layer 1 may not close fully in this crop.")
if labeled_regions.max() > 1:
    largest_label = max(regionprops(labeled_regions), key=lambda region: region.area).label
    oocyte_mask = labeled_regions == largest_label

oocyte_area_um2 = oocyte_mask.sum() / (px_per_um ** 2)
print(f"oocyte mask: {oocyte_mask.sum():,} px enclosed ({oocyte_area_um2:.1f} um^2)")

oocyte_outlines = find_contours(oocyte_mask.astype(float), 0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(PNG_BG)
for ax in axes:
    ax.set_facecolor(PNG_BG)

axes[0].imshow(oocyte_mask, cmap="gray")
for outline in oocyte_outlines:
    axes[0].plot(outline[:, 1], outline[:, 0], color=OOCYTE_COLOR, linewidth=2.5)
axes[0].set_title("oocyte mask, with its boundary highlighted", color=PNG_INK)
axes[0].axis("off")

axes[1].imshow(annotations_rgba)
for outline in oocyte_outlines:
    axes[1].plot(outline[:, 1], outline[:, 0], color=OOCYTE_COLOR, linewidth=2.5)
axes[1].set_title("same boundary, over the original segmentation", color=PNG_INK)
axes[1].axis("off")
plt.show()

## Find transcript dots

Each transcript is a small, bright-green, roughly circular dot. Connected groups of green pixels from the transcript-only layer are grouped into blobs, and each blob's centroid
(center of mass) becomes one transcript's position.

> If blobs are touching in the png, we need to add in a watershed step here.


In [ ]:
red = transcripts_rgba[..., 0].astype(int)
green = transcripts_rgba[..., 1].astype(int)
blue = transcripts_rgba[..., 2].astype(int)
alpha = transcripts_rgba[..., 3]

dot_mask = (alpha > 0) & (green - red >= GREEN_DOMINANCE_MIN) & (green - blue >= GREEN_DOMINANCE_MIN)
labeled_dots = label(dot_mask)
dot_positions = np.array([region.centroid for region in regionprops(labeled_dots)])  # (row, col) per dot

print(f"transcript dots detected: {len(dot_positions)}")

# crop to a window around the oocyte (where the dots we care about are) before showing.
oocyte_ys, oocyte_xs = np.where(oocyte_mask)
pad = 150
y_slice = slice(max(0, oocyte_ys.min() - pad), oocyte_ys.max() + pad)
x_slice = slice(max(0, oocyte_xs.min() - pad), oocyte_xs.max() + pad)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.patch.set_facecolor(PNG_BG)
for ax in axes:
    ax.set_facecolor(PNG_BG)
axes[0].imshow(transcripts_rgba[y_slice, x_slice])
axes[0].set_title("original transcripts layer", color=PNG_INK)
axes[0].axis("off")
axes[1].imshow(dot_mask[y_slice, x_slice], cmap="gray")
axes[1].set_title("detected transcript dots", color=PNG_INK)
axes[1].axis("off")
plt.show()

## Classify each dot as inside or outside the oocyte

Since `dot_positions` and `oocyte_mask` are both already in the same pixel grid, directly look up its positioning for classification

In [ ]:
dot_rows = np.clip(dot_positions[:, 0].round().astype(int), 0, oocyte_mask.shape[0] - 1)
dot_cols = np.clip(dot_positions[:, 1].round().astype(int), 0, oocyte_mask.shape[1] - 1)
dot_is_in_oocyte = oocyte_mask[dot_rows, dot_cols]

print(f"transcript dots inside oocyte: {int(dot_is_in_oocyte.sum())} / {len(dot_positions)}")


## Print overlays, normalize by area

Density normalizes the in-oocyte count by the oocyte's area, since a bigger oocyte will contain more transcripts even at the same expression level.


In [ ]:
oocyte_area_px = int(oocyte_mask.sum())
oocyte_area_um2 = oocyte_area_px / (px_per_um ** 2)
transcripts_in_oocyte = int(dot_is_in_oocyte.sum())
density_per_um2 = transcripts_in_oocyte / oocyte_area_um2 if oocyte_area_um2 else float("nan")

result = {
    "sample_id": sample_id,
    "transcripts_total_detected": len(dot_positions),
    "transcripts_in_oocyte": transcripts_in_oocyte,
    "oocyte_area_px": oocyte_area_px,
    "oocyte_area_um2": oocyte_area_um2,
    "transcript_density_per_um2": density_per_um2,
    "px_per_um": px_per_um,
}
result_df = pd.DataFrame([result])
print(result_df.to_string(index=False))

# Visual QC
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(PNG_BG)
ax.set_facecolor(PNG_BG)
ax.imshow(annotations_rgba)
for outline in find_contours(oocyte_mask.astype(float), 0.5):
    ax.plot(outline[:, 1], outline[:, 0], color=OOCYTE_COLOR, linewidth=1.5)
if len(dot_positions):
    outside = dot_positions[~dot_is_in_oocyte]
    inside = dot_positions[dot_is_in_oocyte]
    ax.scatter(outside[:, 1], outside[:, 0], s=4, color=INK_MUTED, label="outside oocyte")
    ax.scatter(inside[:, 1], inside[:, 0], s=6, color=OOCYTE_COLOR, label="inside oocyte")
ax.legend(loc="upper right", fontsize=8, facecolor=PNG_BG, labelcolor=PNG_INK)
ax.axis("off")
plt.show()

fig.savefig(OUTPUT_DIR / f"{sample_id}_qc_overlay.png", dpi=200, bbox_inches="tight", facecolor=PNG_BG)

In [ ]:
result_df.to_csv(OUTPUT_DIR / f"{sample_id}_oocyte_transcript_counts.csv", index=False)
print(f"saved results to {OUTPUT_DIR / f'{sample_id}_oocyte_transcript_counts.csv'}")


##Compare transcript levels between granulosa cells.

Flood-fill each regions closed mesh (fills every hole, cell-sized
ones included), then subtract whatever territory actually belongs to a different region.

- **Layer 1** = `filled_disk` (already computed in Step 3 -- layer 1's mesh, flood-filled)
  minus the oocyte.
- **Layer 2** = its own mesh, flood-filled the same way, minus the oocyte *and* minus
  layer 1.


In [ ]:
red = annotations_rgba[..., 0].astype(int)
green = annotations_rgba[..., 1].astype(int)
blue = annotations_rgba[..., 2].astype(int)
alpha = annotations_rgba[..., 3]

granulosa_layer2_mask = (
    (alpha >= ALPHA_MIN)
    & (np.abs(red - LAYER2_RGB[0]) <= LAYER2_COLOR_TOLERANCE)
    & (np.abs(green - LAYER2_RGB[1]) <= LAYER2_COLOR_TOLERANCE)
    & (np.abs(blue - LAYER2_RGB[2]) <= LAYER2_COLOR_TOLERANCE)
)
print(f"granulosa layer 2 outline: {granulosa_layer2_mask.sum():,} px matched color {LAYER2_RGB}")

# layer 1's true area: filled_disk (Step 3) is closed_ring's mesh flood-filled -- every
# individual cell interior included -- minus the one hole that's actually the oocyte.
layer1_area_mask = filled_disk & ~oocyte_mask

# layer 2's true area: flood-fill layer 2's own mesh the same way, which also fills in
# everything layer 2 encloses (layer 1 + oocyte) as a side effect -- subtract that back out.
layer2_closed = binary_closing(granulosa_layer2_mask, structure=disk(max(1, round(closing_radius_px))))
layer2_filled = binary_fill_holes(layer2_closed)
layer2_area_mask = layer2_filled & ~oocyte_mask & ~layer1_area_mask

layer1_area_um2 = layer1_area_mask.sum() / (px_per_um ** 2)
layer2_area_um2 = layer2_area_mask.sum() / (px_per_um ** 2)
print(f"granulosa layer 1 area: {layer1_area_um2:.1f} um^2")
print(f"granulosa layer 2 area: {layer2_area_um2:.1f} um^2")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(SURFACE)
for ax in axes:
    ax.set_facecolor(SURFACE)
axes[0].imshow(layer1_area_mask, cmap="gray")
axes[0].set_title("granulosa layer 1 area", color=INK)
axes[0].axis("off")
axes[1].imshow(layer2_area_mask, cmap="gray")
axes[1].set_title("granulosa layer 2 area", color=INK)
axes[1].axis("off")
plt.show()

## Compare transcript density: layer 1 vs. layer 2

same way as we do for the oocyte

In [ ]:
dot_in_layer1 = layer1_area_mask[dot_rows, dot_cols] & ~dot_is_in_oocyte
dot_in_layer2 = layer2_area_mask[dot_rows, dot_cols] & ~dot_is_in_oocyte & ~dot_in_layer1
dot_in_background = ~dot_is_in_oocyte & ~dot_in_layer1 & ~dot_in_layer2

region_counts = {
    "oocyte": int(dot_is_in_oocyte.sum()),
    "granulosa_layer1": int(dot_in_layer1.sum()),
    "granulosa_layer2": int(dot_in_layer2.sum()),
    "background": int(dot_in_background.sum()),
}
region_areas_um2 = {
    "oocyte": oocyte_area_um2,
    "granulosa_layer1": layer1_area_um2,
    "granulosa_layer2": layer2_area_um2,
}

region_df = pd.DataFrame({
    "region": list(region_counts.keys()),
    "transcripts": list(region_counts.values()),
})
region_df["area_um2"] = region_df["region"].map(region_areas_um2)
region_df["density_per_um2"] = region_df["transcripts"] / region_df["area_um2"]
print(region_df.to_string(index=False))

region_df.to_csv(OUTPUT_DIR / f"{sample_id}_density_by_region.csv", index=False)
print(f"saved {OUTPUT_DIR / f'{sample_id}_density_by_region.csv'}")

plotted = region_df[region_df["region"] != "background"]
fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
bars = ax.bar(plotted["region"], plotted["density_per_um2"],
               color=[OOCYTE_COLOR, LAYER1_COLOR, LAYER2_COLOR])
ax.set_ylabel("transcript density (per um^2)", color=INK)
ax.set_title("transcript density by region", color=INK)
ax.tick_params(colors=INK)
for spine in ax.spines.values():
    spine.set_color(GRID)
plt.show()

fig.savefig(OUTPUT_DIR / f"{sample_id}_density_by_region.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
print(f"saved {OUTPUT_DIR / f'{sample_id}_density_by_region.png'}")

## Transcript density vs distance from the oocyte boundary

`distance_transform_edt` gives, for every pixel *outside* the oocyte, its distance to
the nearest oocyte pixel (and 0 for pixels inside the oocyte). Looking each dot up in
that distance map gives a per-dot "how far outside the oocyte is this transcript"
value, in microns. Binning dots by that distance and dividing by how much image area
actually exists at each distance (a thin shell has less area than a thick one) gives a
density curve instead of a raw count curve.

Dashed lines mark roughly where layer 1 ends and layer 2 ends, using the 90th
percentile distance of dots actually assigned to each layer in Step 8 (a robust stand-in
for "the outer edge", since the two layers aren't perfectly circular).


In [ ]:
distance_px = distance_transform_edt(~oocyte_mask)
distance_um = distance_px / px_per_um
dot_distance_um = distance_um[dot_rows, dot_cols]

bin_width_um = 5
max_distance_um = np.percentile(dot_distance_um, 99)
bin_edges = np.arange(0, max_distance_um + bin_width_um, bin_width_um)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

counts, _ = np.histogram(dot_distance_um, bins=bin_edges)
shell_area_um2 = np.array([
    ((distance_um >= lo) & (distance_um < hi)).sum() / (px_per_um ** 2)
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:])
])
density = np.divide(counts, shell_area_um2, out=np.zeros_like(counts, dtype=float), where=shell_area_um2 > 0)

layer1_edge_um = np.percentile(dot_distance_um[dot_in_layer1], 90) if dot_in_layer1.any() else None
layer2_edge_um = np.percentile(dot_distance_um[dot_in_layer2], 90) if dot_in_layer2.any() else None

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
ax.plot(bin_centers, density, marker="o", color=INK)
if layer1_edge_um is not None:
    ax.axvline(layer1_edge_um, color=LAYER1_COLOR, linestyle="--", label="~layer 1 outer edge")
if layer2_edge_um is not None:
    ax.axvline(layer2_edge_um, color=LAYER2_COLOR, linestyle="--", label="~layer 2 outer edge")
ax.set_xlabel("distance from oocyte boundary (um)", color=INK)
ax.set_ylabel("transcript density (per um^2)", color=INK)
ax.set_title("transcript density vs. distance from oocyte", color=INK)
ax.tick_params(colors=INK)
for spine in ax.spines.values():
    spine.set_color(GRID)
ax.legend(facecolor=SURFACE, labelcolor=INK)
plt.show()

fig.savefig(OUTPUT_DIR / f"{sample_id}_density_vs_distance.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)

## Overlay and save


In [ ]:
cell_layer_rgba = np.array(Image.open(CELL_LAYER_FILE).convert("RGBA"))
assert cell_layer_rgba.shape[:2] == annotations_rgba.shape[:2], \
    "Cell_layer.png has different dimensions -- exports must be aligned."

layer1_outlines = find_contours(layer1_area_mask.astype(float), 0.5)
layer2_outlines = find_contours(layer2_area_mask.astype(float), 0.5)

fig, ax = plt.subplots(figsize=(11, 7))
fig.patch.set_facecolor(SURFACE)
ax.imshow(cell_layer_rgba)

for outline in layer2_outlines:
    ax.plot(outline[:, 1], outline[:, 0], color=LAYER2_COLOR, linewidth=1.5)
for outline in layer1_outlines:
    ax.plot(outline[:, 1], outline[:, 0], color=LAYER1_COLOR, linewidth=1.5)
for outline in oocyte_outlines:
    ax.plot(outline[:, 1], outline[:, 0], color=OOCYTE_COLOR, linewidth=2)

ax.scatter(dot_positions[dot_in_background, 1], dot_positions[dot_in_background, 0],
           s=4, color=INK_MUTED, label="background")
ax.scatter(dot_positions[dot_in_layer2, 1], dot_positions[dot_in_layer2, 0],
           s=6, color=LAYER2_COLOR, label="in layer 2")
ax.scatter(dot_positions[dot_in_layer1, 1], dot_positions[dot_in_layer1, 0],
           s=6, color=LAYER1_COLOR, label="in layer 1")
ax.scatter(dot_positions[dot_is_in_oocyte, 1], dot_positions[dot_is_in_oocyte, 0],
           s=8, color=OOCYTE_COLOR, label="in oocyte")

ax.legend(loc="upper right", fontsize=8, facecolor=SURFACE, labelcolor=INK)
ax.set_title(f"{sample_id} -- oocyte + granulosa boundaries, transcripts by region", color=INK)
ax.axis("off")
plt.show()

fig.savefig(OUTPUT_DIR / f"{sample_id}_overlay_on_cell_layer.png", dpi=200, bbox_inches="tight", facecolor=SURFACE)
print(f"saved {OUTPUT_DIR / f'{sample_id}_overlay_on_cell_layer.png'}")

## Download results

`/content` isn't persistent -- it's wiped when the runtime disconnects. This zips up everything in `OUTPUT_DIR` and downloads it to your computer.

In [ ]:
import shutil
from google.colab import files as colab_files
archive_path = shutil.make_archive(f"/content/{sample_id}_results", "zip", OUTPUT_DIR)
colab_files.download(archive_path)
